## Data Cleaning Notebook

1. Cleaning the raw extracted data from airbnb stored at `"../Data/raw/Airbnbdata` to a clean csv ready for analysis and model training
2. Further cleaning after scrabbing extra columns data

In [79]:
import pandas as pd
import json
import re

with open(r"../Data/raw/Airbnb_Scrabbed.json", encoding="utf-8") as f:
    data = json.load(f)
    
if isinstance(data, dict):
    data = [data]

df = pd.json_normalize(data)

df.columns = (
    df.columns
    .str.replace('.', '_')  
    .str.lower()             
)

if 'price_price' in df.columns:
    df['price_price'] = (
        df['price_price']
        .astype(str)
        .str.replace('$', '', regex=False)
        
    )

print("Shape:", df.shape)

Shape: (833, 36)


In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 36 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   thumbnail                                 833 non-null    object 
 1   id                                        833 non-null    object 
 2   title                                     783 non-null    object 
 3   description                               833 non-null    object 
 4   url                                       833 non-null    object 
 5   rating_accuracy                           467 non-null    float64
 6   rating_checking                           467 non-null    float64
 7   rating_cleanliness                        467 non-null    float64
 8   rating_communication                      467 non-null    float64
 9   rating_location                           467 non-null    float64
 10  rating_value                          

#### Get price from json `Price/breakdown/basePrice/description` most accurate 

In [81]:
import numpy as np
def extract_nightly_price_from_str(desc, price=None):
    """Extract nightly price from a string like '3 nights x $120.00' or fallback to price/num_nights."""
    if isinstance(desc, str):
        match = re.search(r'(\d+)\s*nights?\s*x\s*\$(\d+(?:\.\d+)?)', desc)
        if match:
            return float(match.group(2))
        # Fallback: try to divide price by nights if both are available
        nights_match = re.search(r'(\d+)\s*nights?', desc)
        if nights_match and price is not None:
            try:
                nights = int(nights_match.group(1))
                base_total = float(str(price).replace("$", "").replace(",", ""))
                if nights > 0:
                    return round(base_total / nights, 2)
            except Exception:
                pass
    return np.nan
# Apply extraction to the DataFrame
df['nightly_price'] = df.apply(lambda row: extract_nightly_price_from_str(row['price_breakdown_baseprice_description'], row.get('price_breakdown_baseprice_price')), axis=1)


In [82]:

cols_to_keep = [
    "id", "title", "url", "thumbnail",
    "coordinates_latitude", "coordinates_longitude",
    "rating_guestsatisfaction", "rating_reviewscount",
    "rating_accuracy", "rating_cleanliness",
    "rating_value", "rating_location",
    "description",
    "nightly_price"  
]

cols_to_keep = [c for c in cols_to_keep if c in df.columns]
df = df[cols_to_keep].copy()


df.head()

,id,title,url,thumbnail,coordinates_latitude,coordinates_longitude,rating_guestsatisfaction,rating_reviewscount,rating_accuracy,rating_cleanliness,rating_value,rating_location,description,nightly_price
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,Enjoy your stay with Panoramic View of the Giz...,150.00
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,Welcome To Marhaba Pyramids View Hotel✨Wake up...,105.37
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,The place is spacious and can accommodate more...,38.64
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,A designer retreat where ancient soul meets mo...,118.49
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,23.22


#### Rename the columns for clarity

In [83]:
# to rename the columns names
df.rename(columns={
    "coordinates_latitude":               "lat",
    "coordinates_longitude":              "lng",
    "rating_guestsatisfaction":           "rating_overall",
    "rating_reviewscount":                "reviews_count",    
}, inplace=True)
df.head()

,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,rating_value,rating_location,description,nightly_price
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,Enjoy your stay with Panoramic View of the Giz...,150.00
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,Welcome To Marhaba Pyramids View Hotel✨Wake up...,105.37
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,The place is spacious and can accommodate more...,38.64
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,A designer retreat where ancient soul meets mo...,118.49
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,23.22


#### Checking for nulls

In [84]:
print(df.isnull().sum())
print("\n")

id                      0
title                  50
url                     0
thumbnail               0
lat                     2
lng                     2
rating_overall        366
reviews_count         366
rating_accuracy       366
rating_cleanliness    366
rating_value          366
rating_location       366
description             0
nightly_price           0
dtype: int64




In [85]:
df.columns

Index(['id', 'title', 'url', 'thumbnail', 'lat', 'lng', 'rating_overall',
       'reviews_count', 'rating_accuracy', 'rating_cleanliness',
       'rating_value', 'rating_location', 'description', 'nightly_price'],
      dtype='object')

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  833 non-null    object 
 1   title               783 non-null    object 
 2   url                 833 non-null    object 
 3   thumbnail           833 non-null    object 
 4   lat                 831 non-null    float64
 5   lng                 831 non-null    float64
 6   rating_overall      467 non-null    float64
 7   reviews_count       467 non-null    float64
 8   rating_accuracy     467 non-null    float64
 9   rating_cleanliness  467 non-null    float64
 10  rating_value        467 non-null    float64
 11  rating_location     467 non-null    float64
 12  description         833 non-null    object 
 13  nightly_price       833 non-null    float64
dtypes: float64(9), object(5)
memory usage: 91.2+ KB


### Checking duplicates

In [87]:
df.duplicated().sum()

np.int64(0)

### Checking Nulls count

In [88]:
df.isna().sum()

id                      0
title                  50
url                     0
thumbnail               0
lat                     2
lng                     2
rating_overall        366
reviews_count         366
rating_accuracy       366
rating_cleanliness    366
rating_value          366
rating_location       366
description             0
nightly_price           0
dtype: int64

### Add `has_rating` flag according to rating
Keep track of the replaced nulls

In [89]:
df['has_rating'] = df['rating_overall'].notna()

### Nulls Handling

In [90]:
# title
df['title'] = df['title'].fillna('Unknown')

# reviews
df['reviews_count'] = df['reviews_count'].fillna(0)

# drop rows without location
df = df.dropna(subset=['lat', 'lng']).copy()

# ratings
rating_cols = [
    'rating_overall',
    'rating_accuracy',
    'rating_cleanliness',
    'rating_value',
    'rating_location'
]

for col in rating_cols:
    df[col] = df[col].fillna(df[col].mean())

# check
print(df[rating_cols].isna().sum())

rating_overall        0
rating_accuracy       0
rating_cleanliness    0
rating_value          0
rating_location       0
dtype: int64


In [91]:
print(df.isna().sum())
df.head()

id                    0
title                 0
url                   0
thumbnail             0
lat                   0
lng                   0
rating_overall        0
reviews_count         0
rating_accuracy       0
rating_cleanliness    0
rating_value          0
rating_location       0
description           0
nightly_price         0
has_rating            0
dtype: int64


,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,rating_value,rating_location,description,nightly_price,has_rating
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,Enjoy your stay with Panoramic View of the Giz...,150.00,True
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,Welcome To Marhaba Pyramids View Hotel✨Wake up...,105.37,True
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,The place is spacious and can accommodate more...,38.64,True
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,A designer retreat where ancient soul meets mo...,118.49,True
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,23.22,True


#### convert the strings to float

In [92]:
cols = [
    'rating_overall',
    'reviews_count',
    'rating_accuracy',
    'rating_cleanliness',
    'rating_value',
    'rating_location',
    'id'
]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## Further Cleaning 
(After scrabbing bedrooms and bathrooms data and saving to `raw/airbnb_rooms.csv`)

In [101]:
df2 = pd.read_csv('../Data/raw/airbnb_rooms.csv')
df2.head()

,id,bedrooms,bathrooms
0,1292713234154945394,1,1.0
1,1508718511630646313,1,NaN
2,1297327219631789358,1,1.0
3,1606001853199411128,1,1.0
4,1314833467096489875,1,NaN


In [102]:
df2.isnull().sum()

id             0
bedrooms      45
bathrooms    108
dtype: int64

In [103]:
df2['bathrooms'] = df2['bathrooms'].fillna(df2['bathrooms'].mode()[0])
df2['bedrooms'] = df2['bedrooms'].fillna(df2['bedrooms'].mode()[0])
df2.isnull().sum()

id           0
bedrooms     0
bathrooms    0
dtype: int64

In [104]:
df2['bedrooms'] = df2['bedrooms'].replace('0 (studio)', 0)
df2['bedrooms'] = pd.to_numeric(df2['bedrooms'], errors='coerce')

In [105]:
df = df.merge(df2, on="id", how="inner")  

In [106]:
df.head()

,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,rating_value,rating_location,description,nightly_price,has_rating,bedrooms_x,bathrooms_x,bedrooms_y,bathrooms_y
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,Enjoy your stay with Panoramic View of the Giz...,150.00,True,1,1.0,1,1.0
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,Welcome To Marhaba Pyramids View Hotel✨Wake up...,105.37,True,1,1.0,1,1.0
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,The place is spacious and can accommodate more...,38.64,True,1,1.0,1,1.0
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,A designer retreat where ancient soul meets mo...,118.49,True,1,1.0,1,1.0
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,23.22,True,1,1.0,1,1.0


In [107]:
df.to_csv('../Data/raw/airbnb_processed.csv',index=False, encoding="utf-8-sig")